# Agente Simples do Semantic Kernel com plugin de SQL
Agente do Semantic Kernel que se conecta ao azure foundry e usa um plugin para se conectar ao lakehouse endpoint para que possa ler os dados.  



In [21]:
# Instalar e importar dependências
import os, pyodbc, struct, urllib
from dotenv import load_dotenv, dotenv_values 
from azure.identity import ClientSecretCredential
from itertools import chain, repeat
import pandas as pd
import sqlalchemy as sa

# Carregar variáveis do arquivo .env de um diretório específico
env_path = "../../../.env"
load_dotenv(dotenv_path=env_path)

# Alternativa: Carregar .env da raiz do projeto
# project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("")))))
# env_path = os.path.join(project_root, ".env")
# load_dotenv(dotenv_path=env_path)

True

In [25]:
# Crie um service principal no Microsift Entra e obtenha as credenciais
# Referencia: https://youtu.be/IFp1Aingnmw
# Your Azure AD credentials from environment variables

tenant_id = os.getenv("TENANT_ID")
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

if not all([tenant_id, client_id, client_secret]):
    raise ValueError("Por favor, configure as variáveis TENANT_ID, CLIENT_ID e CLIENT_SECRET no arquivo .env")

# Conecta inicializa o ClientSecretCredential
credential = ClientSecretCredential(tenant_id, client_id, client_secret)
# This will remain as is
resource_url = "https://database.windows.net/.default"
# Get the token
token_object = credential.get_token(resource_url)# Retrieve an access token valid to connect to SQL databases

# Definicao da Connection String
sql_endpoint = "agfiegsmmdtefbw5ux6mhctjne-5fuv4ueno5uu3ah6i3bw4u4sxy.datawarehouse.fabric.microsoft.com"
database = "MyAdvWorksLH"

connection_string = f"Driver={{ODBC Driver 18 for SQL Server}};Server={sql_endpoint},1433;Database=f{database};Encrypt=Yes;TrustServerCertificate=No"
params = urllib.parse.quote(connection_string)


In [ ]:

token_as_bytes = bytes(token_object.token, "UTF-8") # Convert the token to a UTF-8 byte string
encoded_bytes = bytes(chain.from_iterable(zip(token_as_bytes, repeat(0)))) # Encode the bytes to a Windows byte string
token_bytes = struct.pack("<i", len(encoded_bytes)) + encoded_bytes # Package the token into a bytes object
attrs_before = {1256: token_bytes}  # Attribute pointing to SQL_COPT_SS_ACCESS_TOKEN to pass access token to the driver

# build the connection
engine = sa.create_engine("mssql+pyodbc:///?odbc_connect={0}".format(params), connect_args={'attrs_before': attrs_before})

SQL_QUERY = """SELECT TOP (100) [AccountKey],
			[ParentAccountKey],
			[AccountCodeAlternateKey],
			[ParentAccountCodeAlternateKey],
			[AccountDescription],
			[AccountType],
			[Operator],
			[CustomMembers],
			[ValueType],
			[CustomMemberOptions]
FROM [MyAdvWorksLH].[dbo].[dimaccount]
"""

# SQL query execution using Pandas
df = pd.read_sql(SQL_QUERY, engine)

    AccountKey  ParentAccountKey  AccountCodeAlternateKey  \
0            1               NaN                        1   
1            2               1.0                       10   
2            3               2.0                      110   
3            4               3.0                     1110   
4            5               3.0                     1120   
..         ...               ...                      ...   
94          97              95.0                     9520   
95          98              95.0                     9530   
96          99              95.0                     9540   
97         100              27.0                     2220   
98         101              51.0                     4200   

    ParentAccountCodeAlternateKey                      AccountDescription  \
0                             NaN                           Balance Sheet   
1                             1.0                                  Assets   
2                            10.0   

In [27]:
df.head()  # Display the first few rows of the DataFrame

,AccountKey,ParentAccountKey,AccountCodeAlternateKey,ParentAccountCodeAlternateKey,AccountDescription,AccountType,Operator,CustomMembers,ValueType,CustomMemberOptions
0,1,NaN,1,NaN,Balance Sheet,None,~,None,Currency,None
1,2,1.0,10,1.0,Assets,Assets,+,None,Currency,None
2,3,2.0,110,10.0,Current Assets,Assets,+,None,Currency,None
3,4,3.0,1110,110.0,Cash,Assets,+,None,Currency,None
4,5,3.0,1120,110.0,Receivables,Assets,+,None,Currency,None


In [ ]:
# Usando somente o PyODBC para conexão e consulta
connection = pyodbc.connect(connection_string, attrs_before=attrs_before)

# Create a cursor object
cursor = connection.cursor()

# Execute a query
cursor.execute(SQL_QUERY)

# Fetch and print results
for row in cursor.fetchall():
    print(row)

# Close the connection
cursor.close()
connection.close()

(1, None, 1, None, 'Balance Sheet', None, '~', None, 'Currency', None)
(2, 1, 10, 1, 'Assets', 'Assets', '+', None, 'Currency', None)
(3, 2, 110, 10, 'Current Assets', 'Assets', '+', None, 'Currency', None)
(4, 3, 1110, 110, 'Cash', 'Assets', '+', None, 'Currency', None)
(5, 3, 1120, 110, 'Receivables', 'Assets', '+', None, 'Currency', None)
(6, 5, 1130, 1120, 'Trade Receivables', 'Assets', '+', None, 'Currency', None)
(7, 5, 1140, 1120, 'Other Receivables', 'Assets', '+', None, 'Currency', None)
(8, 3, 1150, 110, 'Allowance for Bad Debt', 'Assets', '+', None, 'Currency', None)
(9, 3, 1160, 110, 'Inventory', 'Assets', '+', None, 'Currency', None)
(10, 9, 1162, 1160, 'Raw Materials', 'Assets', '+', None, 'Currency', None)
(11, 9, 1164, 1160, 'Work in Process', 'Assets', '+', None, 'Currency', None)
(12, 9, 1166, 1160, 'Finished Goods', 'Assets', '+', None, 'Currency', None)
(13, 3, 1170, 110, 'Deferred Taxes', 'Assets', '+', None, 'Currency', None)
(14, 3, 1180, 110, 'Prepaid Expenses',

NameError: name 'conn' is not defined